In [1]:
from dolfinx import fem, default_scalar_type, mesh as msh
import dolfinx
import ufl
from SPDE_problems import *
import torch
from torch_geometric.data import Data
import torch
from mpi4py import MPI
import pyvista as pv
from Training_utils import train
import torch_geometric
from IPython import display
from FEniCSx_PyTorch_interface import fem_solver

class gat(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.model = torch_geometric.nn.models.GAT(
            in_channels=9, 
            hidden_channels=9, 
            num_layers=10, 
            out_channels=1, 
            v2=True, 
            #dropout=0., 
            act=torch.relu, 
            #norm=torch_geometric.nn.norm.LayerNorm(1),
            add_self_loops=False,
            edge_dim=4,
            residual=False
        )

    def forward(self, data) -> torch.Tensor:
        x, edge_index, edge_attr, upper = data.x, data.edge_index, data.edge_attr, data.upper
        h = self.model(
            x=x,
            edge_index=edge_index,
            edge_attr=edge_attr
        )
        return upper*torch.sigmoid(h)



def train(model, data, optimizer, loss_fn, device):
    model.train()

    data = data.to(device)
    optimizer.zero_grad()

    out = model(data)
    loss = loss_fn(out, data.y)

    loss.backward()
    optimizer.step()

    return loss.item()


def int_to_prblm(idx, mesh):
    if idx == 0:
        return wedge(mesh=mesh)
    if idx == 1:
        return bump(mesh=mesh)
    if idx == 2:
        return lifted_edge(mesh=mesh)
    if idx == 3:
        return cylinder(mesh=mesh)
    if idx == 4:
        return falloff(mesh=mesh)
    if idx == 5:
        return curved_wave(mesh=mesh)
    if idx == 6:
        return curved_waves(mesh=mesh)

def interpolate_expr(expr, Wh):
    f = fem.Function(Wh)
    if expr ==None:
        fem_expr = fem.Expression(fem.Constant(Wh.mesh, default_scalar_type(0)), Wh.element.interpolation_points())
    else:
        fem_expr = fem.Expression(expr, Wh.element.interpolation_points())
    f.interpolate(fem_expr)
    return f

def fs_to_x(fs):
    x = torch.tensor(
        [
            interpolate_expr(fs.eps, fs.Yh).x.array,
            interpolate_expr(fs.b[0], fs.Yh).x.array,
            interpolate_expr(fs.b[1], fs.Yh).x.array,
            interpolate_expr(fs.c, fs.Yh).x.array,
            interpolate_expr(fs.f, fs.Yh).x.array,
            interpolate_expr(ufl.CellDiameter(fs.uh.function_space.mesh), fs.Yh).x.array,
            interpolate_expr(fs.uh, fs.Yh).x.array,
            interpolate_expr(fs.uh.dx(0), fs.Yh).x.array,
            interpolate_expr(fs.uh.dx(1), fs.Yh).x.array
        ],dtype=torch.float32
    )
    return x.T


loss_fn_supervised = torch.nn.MSELoss()



nx = 32
ny = 32
comm = MPI.COMM_WORLD
cell_type = msh.CellType.triangle
prblm_id = 2
mesh = msh.create_unit_square(comm=comm,nx=nx,ny=ny,cell_type=cell_type)

fs = int_to_prblm(prblm_id, mesh)


solver = fem_solver(fs=fs)
def loss_fn_self_supervised(data, y):
    return solver(data)

model = gat()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)


yh_std = fs.yh.x.array.copy()
fs.optimize(max_iter=1000)
yh_opt = fs.yh.x.array.copy()

display.clear_output()

J: 1.5766637168365054e-05
J: 6.73253326292339e-06
J: 2.839166496639431e-06
J: 1.6426038950336954e-06
J: 1.041562053637721e-06
J: 6.58620671061354e-07
J: 4.6353366032062496e-07
J: 3.187828675775473e-07
J: 2.448557760120486e-07
J: 1.7834337603678792e-07
J: 1.3838733467716475e-07
J: 1.0882961277357552e-07
J: 9.028990680476344e-08
J: 7.690696207781221e-08
J: 6.537708091056561e-08
J: 5.5357993581178216e-08
J: 5.042103758168401e-08
J: 4.395242776547851e-08
J: 4.149918083629037e-08
J: 3.620868699303161e-08
J: 3.4301886877519385e-08
J: 3.1594276880464445e-08
J: 2.8526633705541215e-08
J: 2.622065635001947e-08
J: 2.493644567548887e-08
J: 2.353374766286655e-08
J: 2.231274837750352e-08
J: 2.0907166669926732e-08
J: 2.0082668907560928e-08
J: 1.9331095229373918e-08
J: 1.873917568179982e-08
J: 1.809638435569888e-08
J: 1.7514104302805518e-08
J: 1.708349694576806e-08
J: 1.6701389398919398e-08
J: 1.6300214981655764e-08
J: 1.6065326533689563e-08
J: 1.5602367166027802e-08
J: 1.5410978766341417e-08
J: 1.515

KeyboardInterrupt: 

In [ ]:
import torch_geometric

def d_num(num, fs, yh_std, cellfun, delta=1e-8, eps=1e-8):
    delta = 1e-8
    fs.set_weights(yh_std)
    cellfun.interpolate(fs.uh)
    l1 = cellfun.x.array.copy()
    yh_arr = fs.yh.x.array.copy()
    yh_arr[num] += delta

    fs.set_weights(yh_arr)
    cellfun.interpolate(fs.uh)
    l2 = cellfun.x.array.copy()
    dnum = (l2-l1)/delta
    return dnum, abs(dnum)>eps


def relative_position(s, t, pts):
    dx, dy = pts[s]-pts[t]

    dist = np.sqrt(dx**2+dy**2)
    return dist, dx/(dist+1e-16), dy/(dist+1e-16)


for num in range(len(test_set)):
    fs , G = Data_to_solver(num, train = False)
    yh_std = fs.yh.x.array.copy()
    cellfun = fem.Function(fs.Yh)
    pts = msh.compute_midpoints(fs.domain, 2, np.arange(len(fs.yh.x.array)))[:,:2]

    source_nodes = []
    target_nodes = []
    edge_attr = []
    arr = np.arange(len(yh_std))
    cellfun = fem.Function(fs.Yh)
    for i in range(len(yh_std)):
        dnum, mask = d_num(i, fs, yh_std, cellfun, 1e-8, 1e-8)
        print(f"source node: {i}, {len(mask[mask])} targets")
        for j in arr[mask]:
            source_nodes.append(i)
            target_nodes.append(j)
            dist, dx, dy = relative_position(i,j, pts)
            edge_attr.append((dnum[j], dist, dx, dy))
    display.clear_output() 
    edge_index = torch.tensor([source_nodes, target_nodes], dtype=torch.long)
    edge_attr = torch.tensor(edge_attr, dtype=torch.float32).view(-1,4)

    upper = torch.tensor(fs._upper, dtype=torch.float32).view(-1,1)

    Gt = Data(x=G.x, edge_index=edge_index, edge_attr=edge_attr, y=G.y, upper=upper, prblm_id=G.prblm_id, mesh_id=G.mesh_id)
    torch.save(Gt, f"data/test_set_edge_attr/input_values/raw/G_{num}.pt")

In [18]:
from dolfinx import fem, default_scalar_type, mesh as msh
import dolfinx
import ufl
from SPDE_problems import *
import torch
from torch_geometric.data import Data
import torch
from mpi4py import MPI
import pyvista as pv
from Training_utils import train
import torch_geometric
from IPython import display
from FEniCSx_PyTorch_interface import fem_solver
num = 400
num = 1053
num = 0
num = 600
prblm = 0
fs , G = Data_to_solver(prblm, train = False)

from FEniCSx_solver import fem_plotter_grid
delta = 1e-8
yh_std = fs.yh.x.array.copy()
fs.set_weights(yh_std)
fs.set_weights(np.zeros_like(yh_std))
#fs.set_weights(G.y.view(-1).detach().numpy())
l1 = fs.uh.x.array.copy()
yh_arr = fs.yh.x.array.copy()
yh_arr[num] += delta

fs.set_weights(yh_arr)
l2 = fs.uh.x.array.copy()
yh_delta = np.zeros_like(yh_arr)
yh_delta[num] = 1

pv.global_theme.cmap = "coolwarm"
p = pv.Plotter(
    shape=(2, 3),
    groups=[
        ([0,1], [0])
    ]
)

p.subplot(0,0)
grid = fem_plotter_grid(fs.Wh)
fs.set_weights(G.y.view(-1).detach().numpy())
grid.add_data(fs.uh, point=True)
p.add_mesh(grid.grid.warp_by_scalar(),
    scalar_bar_args={
        "title": f"uh"
    },
    show_edges=True)


p.subplot(0,2)
grid = fem_plotter_grid(fs.Wh)
grid.add_data(yh_delta, point=False)
p.add_mesh(grid.grid,
    scalar_bar_args={
        "title": f"yh_delta"
    },
    show_edges=False)

p.camera_position = 'xy'

p.subplot(1,2)
dnum = (l2-l1)/delta
grid = fem_plotter_grid(fs.Wh)
grid.add_data(dnum, point=False)
p.add_mesh(grid.grid,
    scalar_bar_args={
        "title": f"D_yh"
    },
    show_edges=False)

p.camera_position = 'xy'

p.subplot(0,1)
grid = fem_plotter_grid(fs.Wh)
grid.add_data(yh_delta, point=True)
p.add_mesh(grid.grid.warp_by_scalar(factor=1e-1),
    scalar_bar_args={
        "title": f"yh_delta_warp"
    },
    show_edges=True)


p.subplot(1,1)
grid = fem_plotter_grid(fs.Wh)
grid.add_data(dnum, point=True)
p.add_mesh(grid.grid.warp_by_scalar(factor=1e-1),
    scalar_bar_args={
        "title": f"D_yh_warp"
    },
    show_edges=True)



p.show()

Widget(value='<iframe src="http://localhost:49709/index.html?ui=P_0x350dff110_12&reconnect=auto" class="pyvist…

In [8]:
dnum


array([0., 0., 0., ..., 0., 0., 0.], shape=(1120,))

In [9]:
np.arange(len(fs.domain.geometry.x[:,0]))[(fs.domain.geometry.x[:,0]>0.99)&(fs.domain.geometry.x[:,1]==0)]
mesh = fs.domain
arr = fs.yh.x.array.copy()
Dnum = fem.Function(fs.Wh)
Dnum.x.array[:] = dnum
Dnum.name = 'D_yh'

yh = fem.Function(fs.Yh)
yh.x.array[:] = yh_delta
yh.name = 'yh'


In [10]:



with io.XDMFFile(MPI.COMM_WORLD, f"data/models/results/long_range_dependencies_3.xdmf", "w") as xdmf:
    xdmf.write_mesh(mesh)
    xdmf.write_function(u=Dnum)
    xdmf.write_function(u=yh)

In [8]:
from dolfinx import fem, default_scalar_type, mesh as msh
import dolfinx
import ufl
from SPDE_problems import *
import torch
from torch_geometric.data import Data
import torch
from mpi4py import MPI
import pyvista as pv
from Training_utils import train
import torch_geometric
from IPython import display
from FEniCSx_PyTorch_interface import fem_solver
num = 100
prblm = 0
fs , G = Data_to_solver(prblm, train = False)

from FEniCSx_solver import fem_plotter_grid
delta = 1e-8
yh_std = fs.yh.x.array.copy()
fs.set_weights(yh_std)
l1 = fs.uh.x.array.copy()
yh_arr = fs.yh.x.array.copy()
yh_arr += delta

fs.set_weights(yh_arr)
l2 = fs.uh.x.array.copy()
yh_delta = np.zeros_like(yh_arr)
yh_delta[num] = 1

pv.global_theme.cmap = "coolwarm"
p = pv.Plotter(
    shape=(2, 2)
)

p.subplot(0,0)
grid = fem_plotter_grid(fs.Wh)
fs.set_weights(G.y.view(-1).detach().numpy())
grid.add_data(fs.uh, point=False)
p.add_mesh(grid.grid,
    scalar_bar_args={
        "title": f"uh"
    })

p.camera_position = 'xy'

p.subplot(1,0)
grid = fem_plotter_grid(fs.Wh)
grid.add_data(1/(1+np.exp(-(G.y.view(-1).detach().numpy()))), point=False)
p.add_mesh(grid.grid,
    scalar_bar_args={
        "title": f"yh_opt"
    },
    show_edges=False)

p.camera_position = 'xy'

Duh = np.sqrt(G.x[:,8].detach().numpy()**2 + G.x[:,7].detach().numpy()**2)
p.subplot(0,1)
grid = fem_plotter_grid(fs.Wh)
grid.add_data(1/(1+np.exp(-Duh)), point=False)
p.add_mesh(grid.grid,
    scalar_bar_args={
        "title": f"nabla_uh"
    },
    show_edges=False)

p.camera_position = 'xy'

p.subplot(1,1)
dnum = (l2-l1)/delta

grid = fem_plotter_grid(fs.Wh)
grid.add_data(100/(1+np.exp(-dnum)), point=False)
p.add_mesh(grid.grid,
    scalar_bar_args={
        "title": f"D_yh"
    },
    show_edges=False)

p.camera_position = 'xy'



p.show()

Widget(value='<iframe src="http://localhost:50581/index.html?ui=P_0x1095c7110_6&reconnect=auto" class="pyvista…